Import Library

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset, random_split
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from PIL import Image
import json
import os
import random
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import re
from tqdm.auto import tqdm 
import requests
import zipfile
import pickle

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        # Dành cho Mac
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print(f"Using device: {device}")

Using device: mps


In [2]:
COCO_VAL_IMAGES_URL = "http://images.cocodataset.org/zips/val2014.zip"
VQA_QUESTIONS_URL = "https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Val_mscoco.zip"
VQA_ANNOTATIONS_URL = "https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Val_mscoco.zip"

def download_file(url, save_path):
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    block_size = 1024  # 1 Kibibyte
    
    print(f"Downloading {os.path.basename(save_path)}...")
    with open(save_path, 'wb') as file, tqdm(
        desc=os.path.basename(save_path),
        total=total_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for data in response.iter_content(block_size):
            size = file.write(data)
            bar.update(size)

def setup_data():
    #Tạo thư mục data
    base_dir = os.getcwd()
    data_dir = os.path.join(base_dir, 'data')
    img_root = os.path.join(data_dir, 'images')
    
    if not os.path.exists(data_dir):
        os.makedirs(data_dir)
    if not os.path.exists(img_root):
        os.makedirs(img_root)

    print(f"Data Folder: {data_dir}")

    #XỬ LÝ ẢNH
    val2014_dir = os.path.join(img_root, 'val2014')
    zip_path = os.path.join(data_dir, 'val2014.zip')

    if os.path.exists(val2014_dir) and len(os.listdir(val2014_dir)) > 0:
        print("Image existed")
    else:
        if not os.path.exists(zip_path):
            print("Dowloading")
            download_file(COCO_VAL_IMAGES_URL, zip_path)
        
        # Giải nén
        print("Extracting...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(img_root)
        
        
    #XỬ LÝ CÂU HỎI (QUESTIONS)
    target_json = "v2_OpenEnded_mscoco_val2014_questions.json"
    json_path = os.path.join(data_dir, target_json)
    q_zip_path = os.path.join(data_dir, "questions.zip")

    if os.path.exists(json_path):
        print("Question existed")
    else:
        print("Downloading...")
        download_file(VQA_QUESTIONS_URL, q_zip_path)
        
        print("Zip processing...")
        with zipfile.ZipFile(q_zip_path, 'r') as zip_ref:
            zip_ref.extractall(data_dir)

    final_img_dir = val2014_dir if os.path.exists(val2014_dir) else img_root
    
    return final_img_dir, json_path

try:
    IMG_DIR, JSON_FILE = setup_data()

    print("-" * 30)
    print("Processing complete")
    print(f"Image: {IMG_DIR}")
    print(f"JSON: {JSON_FILE}")
    print("-" * 30)
except Exception as e:
    print(f"Error: {e}")

Data Folder: /Users/quangvo/Library/Mobile Documents/com~apple~CloudDocs/juniorYear/DeepLearning/Midterm_523H0173_523H0178/DL_VQA_Project/data
Image existed
Question existed
------------------------------
Processing complete
Image: /Users/quangvo/Library/Mobile Documents/com~apple~CloudDocs/juniorYear/DeepLearning/Midterm_523H0173_523H0178/DL_VQA_Project/data/images/val2014
JSON: /Users/quangvo/Library/Mobile Documents/com~apple~CloudDocs/juniorYear/DeepLearning/Midterm_523H0173_523H0178/DL_VQA_Project/data/v2_OpenEnded_mscoco_val2014_questions.json
------------------------------


In [3]:
class Vocabulary:
    def __init__(self):
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.freq_threshold = 1

    def __len__(self): return len(self.itos)

    @staticmethod
    def tokenizer(text):
        # Chuyển về chữ thường, bỏ dấu câu cơ bản
        return str(text).lower().replace('?', '').replace('.', '').replace(',', '').split()

    def build_vocabulary(self, sentence_list):
        frequencies = Counter()
        idx = 4
        for sentence in sentence_list:
            for word in self.tokenizer(sentence):
                frequencies[word] += 1
        
        # Chỉ thêm từ xuất hiện nhiều hơn threshold
        for word, count in frequencies.items():
            if count >= self.freq_threshold:
                self.stoi[word] = idx
                self.itos[idx] = word
                idx += 1
    
    def numericalize(self, text):
        tokenized_text = self.tokenizer(text)
        return [
            self.stoi.get(token, self.stoi["<UNK>"])
            for token in tokenized_text
        ]

In [4]:
class VQADataset:
    def __init__(self, json_path, img_dir, vocab=None, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        
        # Load dữ liệu JSON
        print(f"Processing JSON...: {os.path.basename(json_path)}...")
        with open(json_path, 'r') as f:
            data_raw = json.load(f)
            # Tùy format JSON mà key chứa câu hỏi có thể khác nhau
            if 'questions' in data_raw:
                self.data = data_raw['questions']
            elif 'annotations' in data_raw:
                self.data = data_raw['annotations']
            else:
                self.data = data_raw

        # Tự động xây dựng từ điển nếu chưa có
        if vocab is None:
            print("Processing Vocab...")
            self.vocab = Vocabulary()
            # Gom toàn bộ câu hỏi để học từ vựng
            all_questions = [item.get('question', '') for item in self.data]
            self.vocab.build_vocabulary(all_questions)
            print(f"Vocab size: {len(self.vocab)} words.")
        else:
            self.vocab = vocab

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        item = self.data[index]
        
        # --- 1. XỬ LÝ ẢNH ---
        # Format tên file ảnh COCO: COCO_val2014_000000xxxxxx.jpg
        img_id = item['image_id']
        img_filename = f"COCO_val2014_{int(img_id):012d}.jpg"
        img_path = os.path.join(self.img_dir, img_filename)
        
        try:
            image = Image.open(img_path).convert("RGB")
        except FileNotFoundError:
            # Fallback: Tạo ảnh đen nếu thiếu file (tránh crash)
            image = Image.new('RGB', (224, 224))
            
        if self.transform:
            image = self.transform(image)

        # --- 2. XỬ LÝ CÂU HỎI ---
        question = item.get('question', '')
        # Chuyển text -> List số (Tensor)
        q_vec = [self.vocab.stoi["<SOS>"]] + self.vocab.numericalize(question) + [self.vocab.stoi["<EOS>"]]
        
        return image, torch.tensor(q_vec)

In [7]:
def collate_fn(batch):
    images, questions = zip(*batch)

    # Stack ảnh thành khối: [Batch, 3, 224, 224]
    images = torch.stack(images, 0)
    
    # Padding câu hỏi cho bằng độ dài nhau: [Batch, Max_Len]
    questions = pad_sequence(questions, batch_first=True, padding_value=0)
    
    return images, questions

In [15]:
BATCH_SIZE = 32

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

if 'IMG_DIR' in globals() and 'JSON_FILE' in globals():
    # Đường dẫn lưu dataset đã chia
    processed_dir = os.path.join(os.path.dirname(JSON_FILE), 'processed')
    if not os.path.exists(processed_dir):
        os.makedirs(processed_dir)
    
    split_file = os.path.join(processed_dir, 'dataset_splits.pkl')
    vocab_file = os.path.join(processed_dir, 'vocab.pkl')
    
    # Kiểm tra xem đã có file lưu chưa
    if os.path.exists(split_file) and os.path.exists(vocab_file):
        print("Loading saved dataset splits...")
        
        # Load vocab
        with open(vocab_file, 'rb') as f:
            vocab = pickle.load(f)
        
        # Load full dataset với vocab đã lưu
        full_dataset = VQADataset(JSON_FILE, IMG_DIR, vocab=vocab, transform=transform)
        
        # Load indices đã chia
        with open(split_file, 'rb') as f:
            split_data = pickle.load(f)
            train_indices = split_data['train_indices']
            val_indices = split_data['val_indices']
            test_indices = split_data['test_indices']
        
        # Tạo lại các Subset với indices đã lưu
        from torch.utils.data import Subset
        train_dataset = Subset(full_dataset, train_indices)
        val_dataset = Subset(full_dataset, val_indices)
        test_dataset = Subset(full_dataset, test_indices)
        
        print("Loaded from saved files")
    else:
        print("Creating new dataset splits...")
        
        # Load toàn bộ dữ liệu
        full_dataset = VQADataset(JSON_FILE, IMG_DIR, transform=transform)
        
        # (80% Train - 10% Val - 10% Test)
        total_count = len(full_dataset)
        train_size = int(0.8 * total_count)
        val_size = int(0.1 * total_count)
        test_size = total_count - train_size - val_size
        
        # Chia dataset
        train_dataset, val_dataset, test_dataset = random_split(
            full_dataset, [train_size, val_size, test_size],
            generator=torch.Generator().manual_seed(42)  # Để reproducible
        )
        
        # Lưu vocab
        with open(vocab_file, 'wb') as f:
            pickle.dump(full_dataset.vocab, f)
        
        # Lưu indices của mỗi split
        split_data = {
            'train_indices': train_dataset.indices,
            'val_indices': val_dataset.indices,
            'test_indices': test_dataset.indices
        }
        with open(split_file, 'wb') as f:
            pickle.dump(split_data, f)
        
        print(f"Saved splits to {processed_dir}")
    
    # Tạo DataLoader
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)
    test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)

    # In thông tin
    print("-" * 30)
    print(f"Vocab Size: {len(full_dataset.vocab)}")
    print(f"Total Data: {len(full_dataset)}")
    print(f"Train Set : {len(train_dataset)}")
    print(f"Val Set   : {len(val_dataset)}")
    print(f"Test Set  : {len(test_dataset)}")


Loading saved dataset splits...
Processing JSON...: v2_OpenEnded_mscoco_val2014_questions.json...
Loaded from saved files
------------------------------
Vocab Size: 11074
Total Data: 214354
Train Set : 171483
Val Set   : 21435
Test Set  : 21436
